---
title: "hft-ipc-bench Benchmark Report"
format:
  pdf:
    toc: true
    number-sections: true
    df-print: kable
execute:
  echo: false
  warning: false
  message: false
  cache: false
  freeze: false
---

# hft-ipc-bench Benchmark Report

## Introduction

This report evaluates several in-host inter-process communication (IPC) mechanisms for a high-frequency trading style market data fan-out workload. The benchmark models a single producer distributing fixed-size 64-byte messages to one or more local consumers. The goal is to compare typical latency, throughput capacity, and fan-out scaling behavior as the number of consumers increases.

The tested implementations are:

- **SHM (lock-free)**: a custom shared-memory ring-buffer implementation using POSIX shared memory and atomic coordination.
- **ZeroMQ PUB/SUB**: a brokerless publish/subscribe baseline over local IPC transport.
- **gRPC streaming**: a server-streaming RPC implementation using Protocol Buffers and gRPC C++.
- **Redis Pub/Sub**: a centralized in-memory Pub/Sub baseline using Redis over a Unix domain socket.

The analysis reads `output/analysis_summary.csv`. Because the benchmark was run inside a VM, tail latency can be affected by virtualization noise, vCPU scheduling, and host-side interference. For this reason, **P50 latency is treated as the primary latency metric**, while P90 and P99 are reported as supporting indicators rather than absolute HFT-grade tail-latency evidence.

## Testing Methodology

### System Requirement

| Item | Specification |
|---|---|
| CPU | Intel(R) Core(TM) i5-12400, 2.5 GHz |
| Memory | 32.0 GB |
| OS | Ubuntu 22.04 LTS running in a VM, x86-64 |
| Compiler | g++ |
| Permission | sudo |

### Dependencies

| Dependency Package | Purpose / Description |
|---|---|
| `build-essential`, `g++`, `make`, `pkg-config` | C++ build toolchain |
| `libzmq3-dev` | ZeroMQ C++ library |
| `libhiredis-dev` | Redis C client library |
| `redis-server` | Redis server-side runtime |
| `libgrpc++-dev` | gRPC C++ library |
| `libprotobuf-dev` | Protobuf runtime |
| `protobuf-compiler` | `protoc` code generator |
| `protobuf-compiler-grpc` | gRPC plugin for `protoc` |
| `python3`, `python3-pip` | Runtime for analysis scripts |

Install command:

```bash
sudo apt-get install -y \
    libzmq3-dev \
    libhiredis-dev redis-server \
    libgrpc++-dev libprotobuf-dev \
    protobuf-compiler protobuf-compiler-grpc \
    build-essential g++ pkg-config
```

Verification commands:

```bash
pkg-config --modversion libzmq
pkg-config --modversion hiredis
pkg-config --modversion grpc++
pkg-config --modversion protobuf

protoc --version
which grpc_cpp_plugin
redis-server --version
```

### Build Process

Each module has an independent Makefile with no cross-dependencies, allowing the implementations to be built separately:

```bash
make -C shm
make -C zeroMQ
make -C grpc
make -C Redis
```

### Test Execution

All benchmark runs are orchestrated by `run_bench.sh`, which starts the required producer, consumer, server, and Redis processes automatically.

```bash
./run_bench.sh
./run_bench.sh --ipc shm --msgs 1000000 --cons 4
```

Post-test analysis can be run with:

```bash
python3 analyze.py --out-dir ./output --plots-dir ./output/plots \
    --ipc-list shm zmq grpc redis \
    --msg-list 100000 1000000 10000000 \
    --cons-list 1 2 4 \
    --mps-list 2000000 5000000 10000000
```

### Test Case Matrix

| Parameter | Values |
|---|---|
| IPC Mechanism | `shm` / `zmq` / `grpc` / `redis` |
| Total Messages (N) | `100,000` / `1,000,000` / `10,000,000` |
| Consumer Count (C) | `1` / `2` / `4` |
| Target Send Rate | `2M` / `5M` / `10M` msg/s |

### Metrics

The main metric is **P50 end-to-end latency**, expressed in microseconds in this notebook. P90 and P99 are included for context. Actual throughput is computed from the number of messages received by the consumer and the elapsed wall-clock time between producer start and consumer completion. For multi-consumer runs, consumer-level measurements are averaged within the same workload configuration before cross-workload aggregation.


In [1]:
import os
from pathlib import Path

Path("/tmp/matplotlib").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

CSV_PATH = Path("output/analysis_summary.csv")
FIG_DIR = Path("output/csv_analysis_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

IPC_ORDER = ["shm", "zmq", "grpc", "redis"]
IPC_LABEL = {
    "shm": "SHM",
    "zmq": "ZeroMQ",
    "grpc": "gRPC",
    "redis": "Redis",
}
LAT_COLS = ["p50_ns", "p90_ns", "p99_ns"]

df = pd.read_csv(CSV_PATH)
df["ipc"] = pd.Categorical(df["ipc"], categories=IPC_ORDER, ordered=True)

numeric_cols = [
    "n_msgs", "n_cons", "target_mps", "consumer_id", "n_samples",
    "p50_ns", "p90_ns", "p99_ns", "p999_ns", "max_ns",
    "avg_jitter_ns", "max_jitter_ns", "elapsed_s", "throughput_mps",
    "n_recv", "n_exp", "loss",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["p50_ns", "p90_ns", "p99_ns", "p999_ns", "max_ns", "avg_jitter_ns", "max_jitter_ns"]:
    df[col.replace("_ns", "_us")] = df[col] / 1_000
    df[col.replace("_ns", "_ms")] = df[col] / 1_000_000

df["throughput_mps_million"] = df["throughput_mps"] / 1_000_000
df["target_mps_million"] = df["target_mps"] / 1_000_000
df["ipc_label"] = df["ipc"].map(IPC_LABEL)


## 1. Latency Percentile Summary by Consumer Count

This table reports P50, P90, and P99 latency by IPC mechanism and consumer count. It does not expand message count or target MPS. Instead, multiple consumers are averaged within each workload configuration, and then the median across message counts and target MPS values is reported.

P50 is the primary comparison metric in this VM-based benchmark. P90 and P99 are included to show tail-latency tendency, but they should be interpreted cautiously because VM scheduling can amplify tail behavior.


In [2]:
latency_points = df.dropna(subset=["p50_us", "p90_us", "p99_us", "n_msgs", "target_mps", "n_cons"]).copy()

latency_avg = (
    latency_points.groupby(
        ["ipc", "ipc_label", "n_msgs", "target_mps", "n_cons"],
        observed=True,
    )
    .agg(
        p50_us_mean=("p50_us", "mean"),
        p90_us_mean=("p90_us", "mean"),
        p99_us_mean=("p99_us", "mean"),
    )
    .reset_index()
)

latency_by_cons = (
    latency_avg.groupby(["ipc", "ipc_label", "n_cons"], observed=True)
    .agg(
        p50_us=("p50_us_mean", "median"),
        p90_us=("p90_us_mean", "median"),
        p99_us=("p99_us_mean", "median"),
        workload_cases=("p50_us_mean", "size"),
    )
    .reset_index()
    .sort_values(["ipc_label", "n_cons"])
)

latency_by_cons[["ipc_label", "n_cons", "workload_cases", "p50_us", "p90_us", "p99_us"]].style.format({
    "n_cons": "{:.0f}",
    "workload_cases": "{:.0f}",
    "p50_us": "{:.2f}",
    "p90_us": "{:.2f}",
    "p99_us": "{:.2f}",
})

,ipc_label,n_cons,workload_cases,p50_us,p90_us,p99_us
0,SHM,1,9,0.11,0.70,62.17
1,SHM,2,9,0.13,0.78,56.65
2,SHM,4,9,0.15,0.87,90.83
3,ZeroMQ,1,9,2483.34,4141.52,4978.41
4,ZeroMQ,2,9,6468.45,11421.50,12430.20
5,ZeroMQ,4,9,51124.22,67436.44,76442.57
6,gRPC,1,9,95464.08,128313.54,173705.87
7,gRPC,2,9,52016.68,87697.98,105501.28
8,gRPC,4,9,29051.46,82972.79,116575.61
9,Redis,1,9,48.91,236.03,373.01


## 2. P50 Latency Scaling by Consumer Count

This figure focuses on the main benchmark question: how typical latency changes as the number of consumers increases. For each `ipc + n_msgs + target_mps + n_cons` configuration, P50 values from multiple consumers are averaged first. Then, for each `ipc + n_cons`, the median across different message counts and target MPS values is used. This avoids overweighting C=2 or C=4 runs simply because they contain more consumer rows.

A lower bar indicates better typical latency. Since the y-axis is logarithmic, differences between mechanisms can span multiple orders of magnitude.


In [3]:
p50_points = df.dropna(subset=["p50_us", "n_msgs", "target_mps", "n_cons"]).copy()
p50_points["n_msgs_label"] = "N=" + p50_points["n_msgs"].astype(int).map("{:,}".format)
p50_points["target_mps_label"] = "MPS=" + p50_points["target_mps"].astype(int).map("{:,}".format)

p50_avg = (
    p50_points.groupby(
        ["ipc", "ipc_label", "n_msgs", "n_msgs_label", "target_mps", "target_mps_million", "target_mps_label", "n_cons"],
        observed=True,
    )
    .agg(
        p50_us_mean=("p50_us", "mean"),
    )
    .reset_index()
)

p50_by_cons = (
    p50_avg.groupby(["ipc", "ipc_label", "n_cons"], observed=True)
    .agg(
        p50_us_median=("p50_us_mean", "median"),
        workload_cases=("p50_us_mean", "size"),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=p50_by_cons,
    x="n_cons",
    y="p50_us_median",
    hue="ipc_label",
    order=sorted(p50_avg["n_cons"].dropna().unique()),
    hue_order=[IPC_LABEL[x] for x in IPC_ORDER],
    errorbar=None,
    ax=ax,
)
ax.set_yscale("log")
ax.set_xlabel("Consumers")
ax.set_ylabel("Median P50 latency across N/MPS workloads (us, log scale)")
ax.set_title("P50 latency scaling by consumer count")
ax.grid(True, axis="y", which="both", alpha=0.25)
ax.legend(title="IPC solution")
fig.tight_layout()
fig.savefig(FIG_DIR / "p50_bar_by_consumers.png", dpi=160, bbox_inches="tight")
plt.show()

p50_by_cons.pivot(index="n_cons", columns="ipc_label", values="p50_us_median")

<Figure size 3000x1800 with 1 Axes>

ipc_label,SHM,ZeroMQ,gRPC,Redis
n_cons,,,,
1,0.106976,2483.337607,95464.077769,48.910801
2,0.133820,6468.448255,52016.682226,71.035234
4,0.148845,51124.220321,29051.460896,99.898981


## 3. P50 Latency vs Target MPS at Fixed Consumer Counts

This plot keeps consumer count fixed in each column and message count fixed in each row, then shows how P50 latency changes as the target send rate increases. Multiple consumers in the same configuration are averaged before plotting, so a C=4 configuration contributes one point per IPC mechanism and target MPS.

This view helps separate fan-out scaling from send-rate pressure. If a mechanism is close to saturation, P50 latency may increase as the target MPS rises. If the producer cannot reach the requested rate, the actual-throughput section should be used together with this plot.


In [4]:
g = sns.relplot(
    data=p50_avg,
    x="target_mps_million",
    y="p50_us_mean",
    hue="ipc_label",
    style="ipc_label",
    row="n_msgs_label",
    col="n_cons",
    kind="line",
    markers=True,
    dashes=False,
    alpha=0.8,
    height=3.2,
    aspect=1.1,
    facet_kws={"sharey": True, "margin_titles": True},
)
g.set_axis_labels("Target MPS (million msg/s)", "Mean P50 across consumers (us, log scale)")
for ax in g.axes.flat:
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.25)
g.fig.suptitle("P50 latency vs target MPS by message count and consumer count", y=1.02)
g.fig.tight_layout()
g.fig.savefig(FIG_DIR / "p50_vs_target_mps_by_n_and_consumers.png", dpi=160, bbox_inches="tight")
plt.show()

<Figure size 3501x2880 with 9 Axes>

## 4. Throughput by Target MPS and Consumer Count

This section reports actual measured throughput. For each `ipc + n_msgs + target_mps + n_cons` configuration, throughput values from multiple consumers are averaged. The table includes the mean throughput in messages per second and the percentage of the target rate achieved.

The bar charts show actual throughput and target-achievement percentage across message counts, target MPS values, and consumer counts. Redis and gRPC may appear close to zero on a linear throughput scale because their measured throughput is much lower than SHM and, in some cases, ZeroMQ.


In [5]:
tp_points = df.dropna(subset=["throughput_mps", "n_msgs", "target_mps", "n_cons"]).copy()
tp_points["n_msgs_label"] = "N=" + tp_points["n_msgs"].astype(int).map("{:,}".format)
tp_points["target_mps_label"] = "Target MPS=" + tp_points["target_mps"].astype(int).map("{:,}".format)
tp_points["throughput_million_mps"] = tp_points["throughput_mps"] / 1_000_000

tp_avg = (
    tp_points.groupby(
        ["ipc", "ipc_label", "n_msgs", "n_msgs_label", "target_mps", "target_mps_million", "target_mps_label", "n_cons"],
        observed=True,
    )
    .agg(
        throughput_mps_mean=("throughput_mps", "mean"),
        throughput_mps_min=("throughput_mps", "min"),
        throughput_mps_max=("throughput_mps", "max"),
    )
    .reset_index()
)
tp_avg["throughput_million_mps_mean"] = tp_avg["throughput_mps_mean"] / 1_000_000
tp_avg["throughput_million_mps_min"] = tp_avg["throughput_mps_min"] / 1_000_000
tp_avg["throughput_million_mps_max"] = tp_avg["throughput_mps_max"] / 1_000_000
tp_avg["target_achievement_pct"] = tp_avg["throughput_mps_mean"] / tp_avg["target_mps"] * 100

tp_table = tp_avg[
    [
        "ipc_label", "n_msgs", "target_mps", "n_cons",
        "throughput_mps_mean", "target_achievement_pct",
    ]
].sort_values(["ipc_label", "n_msgs", "target_mps", "n_cons"])

tp_table.style.format({
    "n_msgs": "{:,}",
    "target_mps": "{:,}",
    "throughput_mps_mean": "{:,.0f}",
    "target_achievement_pct": "{:.2f}%",
})

,ipc_label,n_msgs,target_mps,n_cons,throughput_mps_mean,target_achievement_pct
0,SHM,"100,000","2,000,000",1,"1,997,493",99.87%
1,SHM,"100,000","2,000,000",2,"1,995,842",99.79%
2,SHM,"100,000","2,000,000",4,"1,998,074",99.90%
3,SHM,"100,000","5,000,000",1,"4,955,025",99.10%
4,SHM,"100,000","5,000,000",2,"4,979,148",99.58%
5,SHM,"100,000","5,000,000",4,"4,957,550",99.15%
6,SHM,"100,000","10,000,000",1,"9,909,354",99.09%
7,SHM,"100,000","10,000,000",2,"9,889,857",98.90%
8,SHM,"100,000","10,000,000",4,"9,918,799",99.19%
9,SHM,"1,000,000","2,000,000",1,"2,001,166",100.06%


In [6]:
g = sns.catplot(
    data=tp_avg,
    x="n_cons",
    y="throughput_million_mps_mean",
    hue="ipc_label",
    row="n_msgs_label",
    col="target_mps_label",
    kind="bar",
    order=sorted(tp_avg["n_cons"].dropna().unique()),
    hue_order=[IPC_LABEL[x] for x in IPC_ORDER],
    errorbar=None,
    height=3.2,
    aspect=1.15,
    margin_titles=True,
)
g.set_axis_labels("Consumers", "Mean actual throughput (million msg/s)")
for ax in g.axes.flat:
    ax.grid(True, axis="y", alpha=0.25)
g.fig.suptitle("Mean actual throughput by message count, target MPS, and consumer count", y=1.02)
g.fig.tight_layout()
g.fig.savefig(FIG_DIR / "throughput_bar_by_n_mps_consumers.png", dpi=160, bbox_inches="tight")
plt.show()

<Figure size 3645x2880 with 9 Axes>

In [7]:
g = sns.catplot(
    data=tp_avg,
    x="n_cons",
    y="target_achievement_pct",
    hue="ipc_label",
    row="n_msgs_label",
    col="target_mps_label",
    kind="bar",
    order=sorted(tp_avg["n_cons"].dropna().unique()),
    hue_order=[IPC_LABEL[x] for x in IPC_ORDER],
    errorbar=None,
    height=3.2,
    aspect=1.15,
    margin_titles=True,
)
g.set_axis_labels("Consumers", "Actual / target throughput (%)")
for ax in g.axes.flat:
    ax.axhline(100, color="black", linestyle=":", linewidth=1)
    ax.grid(True, axis="y", alpha=0.25)
g.fig.suptitle("Throughput target achievement by message count, target MPS, and consumer count", y=1.02)
g.fig.tight_layout()
g.fig.savefig(FIG_DIR / "throughput_target_achievement_by_n_mps_consumers.png", dpi=160, bbox_inches="tight")
plt.show()

<Figure size 3645x2880 with 9 Axes>

## 5. Throughput vs P50 Latency

This scatter plot relates capacity and typical latency. The x-axis is actual measured throughput, and the y-axis is P50 latency. The ideal region is toward the lower-right corner: high throughput with low typical latency.

Because the x-axis uses actual throughput rather than target MPS, mechanisms that cannot reach the requested send rate will appear further left. This is important for interpreting Redis and gRPC results: a low latency number is less meaningful if the achieved throughput is far below the target workload.


In [8]:
plot_df = df.dropna(subset=["throughput_mps_million", "p50_us", "n_msgs"]).copy()

fig, ax = plt.subplots(figsize=(12, 7))
sns.scatterplot(
    data=plot_df,
    x="throughput_mps_million",
    y="p50_us",
    hue="ipc_label",
    style="ipc_label",
    size="n_msgs",
    sizes=(35, 220),
    alpha=0.75,
    ax=ax,
)
ax.set_yscale("log")
ax.set_xlabel("Actual throughput (million msg/s)")
ax.set_ylabel("P50 latency (us, log scale)")
ax.set_title("Throughput vs P50 latency")
ax.legend(title="IPC / N", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "throughput_vs_p50.png", dpi=160, bbox_inches="tight")
plt.show()

<Figure size 3600x2100 with 1 Axes>

## 6. Result Summary and Conclusion

### Executive Summary

**Fastest Overall: SHM**

The custom lock-free shared-memory implementation dominated the latency results, with an overall median P50 of about **0.14 us** and maximum measured throughput of about **10.0M msg/s**. It also completed the dataset with **zero message loss**.

**Best Non-SHM Throughput: ZeroMQ**

ZeroMQ achieved the strongest non-SHM throughput, with median actual throughput around **2.0M msg/s** and a peak around **4.36M msg/s**. However, it showed much higher latency than SHM and suffered message loss under overloaded configurations.

**Slowest Typical Latency: gRPC**

gRPC had the highest typical latency in this workload, with an overall median P50 of about **52 ms**. Its RPC stack, serialization path, stream scheduling, and server-side queuing make it a poor fit for ultra-low-latency local fan-out.

### Key Findings

- **SHM is the clear winner for in-host HFT-style fan-out.** It stays in the sub-microsecond P50 range across consumer counts and is the only mechanism that combines very low latency with multi-million-message-per-second throughput.
- **Consumer count is a first-class scaling dimension.** The benchmark is not only a single-consumer latency test; it measures how each mechanism behaves as fan-out increases from C=1 to C=2 and C=4.
- **ZeroMQ is the strongest general messaging baseline for throughput, but not for deterministic latency.** It can move far more messages than Redis or gRPC, but its latency increases sharply with more consumers and high target rates, and overloaded cases show loss.
- **Redis Pub/Sub has relatively low non-SHM P50 latency in this dataset but very limited throughput.** Its centralized server, RESP parsing, Pub/Sub fan-out, and per-message command/reply path make it unsuitable for the requested 2M/5M/10M msg/s target rates.
- **gRPC is convenient but not latency-competitive here.** It is valuable for typed service communication, but its abstraction cost is too high for this 64-byte local market-data fan-out workload.
- **VM results should be interpreted conservatively.** P50 is the primary metric in this report. P90/P99 are useful for trend inspection, but VM scheduling and host interference can distort tail-latency measurements.

### Recommendations

- **Use SHM** when the objective is minimum latency and maximum local throughput, and when the application can tolerate custom shared-memory lifecycle and synchronization code.
- **Use ZeroMQ** when a standard messaging library is preferred and throughput matters more than sub-microsecond latency, while carefully handling high-load loss behavior.
- **Use Redis Pub/Sub** only as a centralized baseline or for operational simplicity at much lower message rates; it is not a good fit for HFT-rate local fan-out.
- **Use gRPC** for service APIs and typed streaming interfaces, not for ultra-low-latency market-data distribution.
- **Repeat on bare metal** for final HFT claims. A bare-metal Linux run with CPU pinning, isolated cores, and hardware performance counters would provide more reliable P99 and micro-architectural measurements.

